
# Stack Overflow Developer Survey 2011 - canonical (not yet encoded) version

**Purpose:** one year of a shared multi-year dataset. This notebook produces `canonical_2011.csv`: shared column names, original text answers kept as they are for single-answer questions, multi-answer questions stored as `;`-separated lists (the format 2016+ files use), and the language write-ins cleaned into canonical names with a count. **No one-hot encoding, no imputation, no scaling, no rare-level merging** - those are learned from the data and belong in the pooled pipeline, fitted on the training years only.

**Rules followed:** nothing assumed until printed; row counts logged at every drop; one action per cell; every mapping shown as a table.

**Gate 1 is the same load and normalize steps as the previous notebook** (including `keep_default_na=False`, so real answers such as `None` survive).


## Gate 1 - Load and normalize

### Cell 1.0 - Imports, file path, decision log
**Purpose:** set up the environment and find the file. The path tries `data/2011/results.csv` first, then `2011_results.csv` in the working directory.
**Decision made:** none. `decision_log` collects every transformation so the final log (Gate 7) is built from what actually ran.

In [1]:
import csv
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 200)

CANDIDATES = [Path("data/2011/results.csv"), Path("2011_results.csv")]
PATH = next((p for p in CANDIDATES if p.exists()), None)
if PATH is None:
    raise FileNotFoundError("none of the candidate paths exist: " + ", ".join(str(p) for p in CANDIDATES))
print("using file:", PATH, "| size in bytes:", PATH.stat().st_size)
print("pandas version:", pd.__version__)

decision_log = []

def log(step, text):
    decision_log.append((step, text))
    print("[LOG]", step, "-", text)

using file: data\2011\results.csv | size in bytes: 939640
pandas version: 3.1.0


### Cell 1.1 - Detect the file encoding
**Purpose:** `read_csv` defaults to UTF-8 and would fail on this file, so check before loading.
**Decision made:** use UTF-8 if the whole file decodes, otherwise `cp1252`, and print the byte that forced the fallback.

In [2]:
raw_bytes = PATH.read_bytes()
non_ascii = sum(1 for b in raw_bytes if b > 127)
print("total bytes:", len(raw_bytes), "| non-ASCII bytes:", non_ascii)
try:
    raw_bytes.decode("utf-8")
    ENCODING = "utf-8"
    print("utf-8 decodes the whole file cleanly")
except UnicodeDecodeError as err:
    ENCODING = "cp1252"
    print("utf-8 fails at byte", err.start, "- surrounding bytes:", raw_bytes[max(0, err.start - 25):err.start + 10])
    print("falling back to cp1252")
log("1.1 encoding", "file read with encoding=" + ENCODING + " (" + str(non_ascii) + " non-ASCII byte(s) in the file)")

total bytes: 939640 | non-ASCII bytes: 1
utf-8 fails at byte 926937 - surrounding bytes: b'u need a box for OS=UNIX \xc3,,So happ'
falling back to cp1252
[LOG] 1.1 encoding - file read with encoding=cp1252 (1 non-ASCII byte(s) in the file)


### Cell 1.2 - Load with `header=0`, without pandas' automatic NA parsing
**Purpose:** load the CSV exactly as written.
**Decision made:** read every cell as text (`dtype=str`) and set `keep_default_na=False, na_values=[]`. By default pandas converts the literal strings `None`, `N/A`, `NA` and `null` to NaN. In this survey `None` is a real answer (for example the "None" language checkbox), so default parsing would silently erase it. Genuine blanks are converted in Cell 1.4 instead, as your gotcha list prescribes.

In [3]:
df = pd.read_csv(PATH, header=0, encoding=ENCODING, dtype=str, keep_default_na=False, na_values=[])
print("shape after load (rows, columns):", df.shape)

header_looks_like_ints = all(str(c).isdigit() for c in df.columns)
print("column names are all integers (header consumed as data)?", header_looks_like_ints)
if header_looks_like_ints:
    raise ValueError("header row was consumed as data - reload")
print("first 3 column names:", list(df.columns[:3]))
log("1.2 load", "loaded " + str(df.shape[0]) + " rows x " + str(df.shape[1]) + " columns, all as text, default NA parsing off")

shape after load (rows, columns): (2814, 65)
column names are all integers (header consumed as data)? False
first 3 column names: ['What Country or Region do you live in?', 'Which US State or Territory do you live in?', 'How old are you?']
[LOG] 1.2 load - loaded 2814 rows x 65 columns, all as text, default NA parsing off


### Cell 1.3 - Cross-check the load with Python's `csv` module
**Purpose:** confirm pandas did not drop, split or merge rows, and find which header cells are truly empty (pandas renames those `Unnamed: n`, so this is the reliable list).
**Decision made:** none. This only produces `blank_header_positions`, used later to map multi-select spans.

In [4]:
with open(PATH, encoding=ENCODING, newline="") as fh:
    csv_rows = list(csv.reader(fh))
print("rows seen by csv module (header + data):", len(csv_rows))
print("distinct row lengths:", sorted(set(len(r) for r in csv_rows)))
assert len(csv_rows) - 1 == len(df), "pandas and csv module disagree on row count"
assert all(len(r) == df.shape[1] for r in csv_rows), "ragged rows found"

header_raw = csv_rows[0]
blank_header_positions = [i for i, h in enumerate(header_raw) if h.strip() == ""]
print("header cells that are empty:", len(blank_header_positions), "of", len(header_raw))
print("first 10 empty header positions:", blank_header_positions[:10])

rows seen by csv module (header + data): 2815
distinct row lengths: [65]
header cells that are empty: 46 of 65
first 10 empty header positions: [9, 10, 11, 12, 13, 14, 16, 17, 18, 19]


### Cell 1.4 - Strip whitespace from column names
**Purpose:** remove leading/trailing spaces from headers, as in your gotcha list.
**Decision made:** `df.columns = df.columns.astype(str).str.strip()`. Cell output lists exactly which names changed.

In [5]:
changed = [(i, repr(c)) for i, c in enumerate(df.columns) if c != c.strip()]
print("column names with leading/trailing whitespace:", changed)
df.columns = df.columns.astype(str).str.strip()
print("names still with whitespace after strip:", sum(c != c.strip() for c in df.columns))
print("duplicate column names:", df.columns[df.columns.duplicated()].tolist())
log("1.4 strip", str(len(changed)) + " column name(s) had surrounding whitespace and were stripped")

column names with leading/trailing whitespace: [(63, "'In the last 12 months, how much money have you spent on personal technology-related purchases? '")]
names still with whitespace after strip: 0
duplicate column names: []
[LOG] 1.4 strip - 1 column name(s) had surrounding whitespace and were stripped


### Cell 1.5 - Normalize blank cells to NaN
**Purpose:** turn empty and whitespace-only cells into NaN, once.
**Decision made:** `df.replace(r"^\s*$", np.nan, regex=True)`. The cell counts empty and whitespace-only cells first, then checks that the NaN count rose by exactly that amount, so we know the replacement actually happened. Literal text such as `None` or `N/A` is **not** touched here; it is listed so you can see it. These counts still include row 0, which Cell 1.6 separates: in col 41 the 21 cells are 20 respondent answers plus the option label "None".

In [6]:
empty_exact = int((df == "").sum().sum())
ws_only = int(df.apply(lambda s: s.str.match(r"^\s+$")).sum().sum())
na_before = int(df.isna().sum().sum())
print("cells exactly empty:", empty_exact, "| whitespace-only (non-empty):", ws_only, "| NaN before:", na_before)

df = df.replace(r"^\s*$", np.nan, regex=True)

na_after = int(df.isna().sum().sum())
print("NaN after:", na_after, "| increase:", na_after - na_before)
assert na_after - na_before == empty_exact + ws_only, "NaN count did not move by the expected amount"
print("total cells:", df.size, "| share missing: %.1f%%" % (100 * na_after / df.size))

literals = ["None", "N/A", "NA", "n/a", "null", "NULL", "NaN", "nan"]
found = []
for i, col in enumerate(df.columns):
    n = int(df.iloc[:, i].isin(literals).sum())
    if n:
        found.append((i, col[:50], n))
print("literal NA-like text kept as real values (not converted):")
for item in found:
    print("   col", item[0], "|", item[1], "|", item[2], "cells")
log("1.5 blanks", str(empty_exact + ws_only) + " empty/whitespace cells converted to NaN; " + str(sum(f[2] for f in found)) + " literal 'None'-style cells kept as text")

cells exactly empty: 117089 | whitespace-only (non-empty): 0 | NaN before: 0
NaN after: 117089 | increase: 117089
total cells: 182910 | share missing: 64.0%
literal NA-like text kept as real values (not converted):
   col 41 | Unnamed: 41 | 21 cells
   col 64 | Which of our sites do you frequent most? | 14 cells
[LOG] 1.5 blanks - 117089 empty/whitespace cells converted to NaN; 35 literal 'None'-style cells kept as text


### Cell 1.6 - Separate the option-label row
**Purpose:** your brief says the file has a second header-like row. Inspect row 0 of the data to see what it is before touching it.
**Decision made (applied because you asked for all gates; say so if you want it reversed):** row 0 is fully populated and holds the option label for each multi-select column (`Influencer`, `Java`, `iPhone`, ...), with the word `Response` for single-select columns. It is not a respondent, so it is moved out of `df` into `option_labels` (nothing is lost) and the move is logged as a row removal. The original index is kept, so index `n` always means data row `n` of the file.

In [7]:
first_row = df.iloc[0]
print("non-empty cells in row 0:", int(first_row.notna().sum()), "of", df.shape[1])
print("cells equal to 'Response':", int((first_row == "Response").sum()))
print("labels in row 0 other than 'Response', in column order:")
for i, v in enumerate(first_row):
    if v != "Response":
        print("   col", i, "->", v)

rows_before = len(df)
option_labels = df.iloc[0].copy()
df = df.iloc[1:].copy()
print("rows before:", rows_before, "| rows after:", len(df))
assert len(df) == rows_before - 1
log("1.6 option row", "row 0 moved to option_labels (rows " + str(rows_before) + " -> " + str(len(df)) + ")")

non-empty cells in row 0: 65 of 65
cells equal to 'Response': 14
labels in row 0 other than 'Response', in column order:
   col 8 -> Influencer
   col 9 -> Recommender
   col 10 -> Approver
   col 11 -> Purchaser
   col 12 -> Check Writer
   col 13 -> No Involvement
   col 14 -> I'm a Seller
   col 15 -> Hardware
   col 16 -> Servers
   col 17 -> Software
   col 18 -> User Equipment: Monitors, PCs, Laptops
   col 19 -> Consultants
   col 20 -> Other
   col 21 -> <$10,000
   col 22 -> $10,001 - $25,000
   col 23 -> $25,001 - $40,000
   col 24 -> $41,000 - $75,000
   col 25 -> $75,001 - $100,000
   col 26 -> $100,001 - $150,000
   col 27 -> >$150,000
   col 28 -> Don't know
   col 30 -> Java
   col 31 -> JavaScript
   col 32 -> CSS
   col 33 -> PHP
   col 34 -> Python
   col 35 -> Ruby
   col 36 -> SQL
   col 37 -> C#
   col 38 -> C++
   col 39 -> C
   col 40 -> Perl
   col 41 -> None
   col 42 -> other (please specify)
   col 46 -> iPhone
   col 47 -> Android
   col 48 -> Blackberry
   

### Cell 1.7 - Fully empty rows
**Purpose:** count respondents with no answers at all. Nothing is dropped in Gate 1; Gate 2 proposes the criteria.
**Decision made:** none.

In [8]:
filled_per_row = df.notna().sum(axis=1)
fully_empty = int((filled_per_row == 0).sum())
print("data rows:", len(df))
print("fully empty rows:", fully_empty, "| dropped in Gate 1: 0")
print("filled cells per row - lowest values (value : number of rows):")
print(filled_per_row.value_counts().sort_index().head(8).to_string())
only_country = int(((filled_per_row == 1) & df.iloc[:, 0].notna()).sum())
print("rows whose ONLY filled cell is Country (col 0):", only_country, "- candidates for Gate 2, not dropped")
log("1.7 empty rows", str(fully_empty) + " fully empty rows found; 0 dropped; " + str(only_country) + " rows have only Country filled (Gate 2 decision)")

data rows: 2813
fully empty rows: 0 | dropped in Gate 1: 0
filled cells per row - lowest values (value : number of rows):
1     73
2     11
4     68
5     15
8      8
9     24
10    10
11     4
rows whose ONLY filled cell is Country (col 0): 73 - candidates for Gate 2, not dropped
[LOG] 1.7 empty rows - 0 fully empty rows found; 0 dropped; 73 rows have only Country filled (Gate 2 decision)


### Cell 1.8 - Indexed column list
**Purpose:** print every column with its index, header text, the option label from row 0, and how many rows have a value. Columns with an empty header are continuations of the question to their left, so they show which question they belong to.
**Decision made:** none. `col_index` is kept for the later gates.

In [9]:
records = []
parent_idx = None
for i, col in enumerate(df.columns):
    is_blank = i in blank_header_positions
    if not is_blank:
        parent_idx = i
    n_filled = int(df.iloc[:, i].notna().sum())
    records.append({
        "idx": i,
        "header": col[:62] if not is_blank else "(blank - continues col " + str(parent_idx) + ")",
        "row0_label": option_labels.iloc[i],
        "n_filled": n_filled,
        "pct_missing": round(100 * (1 - n_filled / len(df)), 1),
    })
col_index = pd.DataFrame(records)
print(col_index.to_string(index=False))

 idx                                                         header                             row0_label  n_filled  pct_missing
   0                         What Country or Region do you live in?                               Response      2813          0.0
   1                    Which US State or Territory do you live in?                               Response      1029         63.4
   2                                               How old are you?                               Response      2729          3.0
   3       How many years of IT/Programming experience do you have?                               Response      2729          3.0
   4          How would you best describe the industry you work in?                               Response      2729          3.0
   5                 Which best describes the size of your company?                               Response      2646          5.9
   6         Which of the following best describes your occupation?                       

### Cell 1.9 - Gate 1 summary
**Purpose:** final shape and the log so far.
**Decision made:** none. Gate 2 follows.

In [10]:
print("final shape (rows, columns):", df.shape)
print("original data rows in file:", len(csv_rows) - 1, "| rows in df:", len(df), "| rows held in option_labels: 1")
print()
print("decision log so far:")
for step, text in decision_log:
    print("  ", step, "|", text)

final shape (rows, columns): (2813, 65)
original data rows in file: 2814 | rows in df: 2813 | rows held in option_labels: 1

decision log so far:
   1.1 encoding | file read with encoding=cp1252 (1 non-ASCII byte(s) in the file)
   1.2 load | loaded 2814 rows x 65 columns, all as text, default NA parsing off
   1.4 strip | 1 column name(s) had surrounding whitespace and were stripped
   1.5 blanks | 117089 empty/whitespace cells converted to NaN; 35 literal 'None'-style cells kept as text
   1.6 option row | row 0 moved to option_labels (rows 2814 -> 2813)
   1.7 empty rows | 0 fully empty rows found; 0 dropped; 73 rows have only Country filled (Gate 2 decision)


## Gate 2 - Freeze names, choices, junk rows

### Cell 2.0 - Raw column names and `CONFIG`
**Purpose:** refer to columns by name from here on, each checked against a keyword from its question text. `CONFIG` holds the only two row-level choices.
**Decision made:** same row choices as the previous notebook: drop junk rows and rows with no usable salary band, so the row count stays 2,150 and the two files stay comparable. Change `drop_student_unemployed` to keep those rows (their `ConvertedCompYearly` would then be empty).

In [11]:
import re
import math
import datetime

RAW = list(df.columns)
IDX = {"country": 0, "state": 1, "age": 2, "exp": 3, "industry": 4, "size": 5, "occ": 6, "rec": 7,
       "project": 29, "os": 43, "sat": 44, "salary": 45, "spend": 63, "site": 64}
COL = {k: RAW[v] for k, v in IDX.items()}
KEYWORD = {"country": "country", "state": "us state", "age": "how old", "exp": "years of it", "industry": "industry",
           "size": "size of your company", "occ": "occupation", "rec": "recommendation", "project": "project",
           "os": "operating system", "sat": "satisfaction", "salary": "compensation", "spend": "spent", "site": "sites"}
for k, col in COL.items():
    assert KEYWORD[k] in col.lower(), "column " + repr(col) + " does not contain " + repr(KEYWORD[k])
    print("ok  idx", IDX[k], "|", k, "|", col[:70])

BLOCKS = {"purchase_involvement": (8, 15), "purchase_types": (15, 21), "budget": (21, 29), "language": (30, 43), "products": (46, 63)}
for key, (a, b) in BLOCKS.items():
    print("block", key, "| cols", a, "to", b - 1, "|", RAW[a][:70])

CONFIG = {"drop_student_unemployed": True, "top_band_midpoint": 150000}
print(CONFIG)

row_log = [("1.6 option-label row moved out of data", len(csv_rows) - 1, len(df))]
def log_rows(step, before, after):
    row_log.append((step, before, after))
    print("[ROWS]", step, ":", before, "->", after, "(removed", before - after, ")")

ok  idx 0 | country | What Country or Region do you live in?
ok  idx 1 | state | Which US State or Territory do you live in?
ok  idx 2 | age | How old are you?
ok  idx 3 | exp | How many years of IT/Programming experience do you have?
ok  idx 4 | industry | How would you best describe the industry you work in?
ok  idx 5 | size | Which best describes the size of your company?
ok  idx 6 | occ | Which of the following best describes your occupation?
ok  idx 7 | rec | How likely is it that a recommendation you make will be acted upon?
ok  idx 29 | project | What type of project are you developing?
ok  idx 43 | os | What operating system do you use the most?
ok  idx 44 | sat | Please rate your job/career satisfaction
ok  idx 45 | salary | Including bonus, what is your annual compensation in USD?
ok  idx 63 | spend | In the last 12 months, how much money have you spent on personal techn
ok  idx 64 | site | Which of our sites do you frequent most?
block purchase_involvement | cols 8 to 14 | W

### Cell 2.1 - Junk rows: propose
**Purpose:** respondents who answered nothing except Country/State (US State is routing, not an answer). **Decision made:** none yet.

In [12]:
other_cols = [c for c in df.columns if c not in (COL["country"], COL["state"])]
answered_other = df[other_cols].notna().sum(axis=1)
is_junk = answered_other == 0
print("rows the rule would remove:", int(is_junk.sum()), "| of those with a salary band:", int((is_junk & df[COL["salary"]].notna()).sum()))

rows the rule would remove: 84 | of those with a salary band: 0


### Cell 2.2 - Junk rows: apply
**Purpose / decision:** drop them (none has a salary). Counts logged.

In [13]:
rows_before = len(df)
df = df.loc[~is_junk].copy()
log_rows("2.2 junk rows (only Country/State answered)", rows_before, len(df))
log("2.2 junk rows", str(rows_before - len(df)) + " rows removed")

[ROWS] 2.2 junk rows (only Country/State answered) : 2813 -> 2729 (removed 84 )
[LOG] 2.2 junk rows - 84 rows removed


## Gate 3 - Salary

### Cell 3.1 - Band table
**Purpose:** the 2011 answer is a text band, not a number. This table fixes the order, the bounds and the midpoint used to make an annual-USD number.
**Decision made:** `CompTotal` keeps the original band text; `ConvertedCompYearly` is the band midpoint in annual USD (2011 asks "including bonus"). The open-ended `>$140,000` band takes `CONFIG["top_band_midpoint"]`. The shared-schema notes say pay before 2019 is base salary only, so 2011 is not strictly comparable with 2019+ and the band-midpoint construction is a 2011-2015 convention.

In [14]:
STUDENT_LABEL = "Student / Unemployed"
BANDS = [
    ("<$20,000", 0, 20000, 10000), ("$20,000 - $40,000", 20000, 40000, 30000), ("$40,000 - $60,000", 40000, 60000, 50000),
    ("$60,000 - $80,000", 60000, 80000, 70000), ("$80,000 - $100,000", 80000, 100000, 90000),
    ("$100,000 - $120,000", 100000, 120000, 110000), ("$120,000 - $140,000", 120000, 140000, 130000),
    (">$140,000", 140000, None, CONFIG["top_band_midpoint"]),
]
observed = set(df[COL["salary"]].dropna().unique())
known = set(b[0] for b in BANDS) | {STUDENT_LABEL}
assert observed <= known, sorted(observed - known)
counts = df[COL["salary"]].value_counts()
print(pd.DataFrame([[b[0], int(counts.get(b[0], 0)), b[1], b[2], b[3]] for b in BANDS] + [[STUDENT_LABEL, int(counts.get(STUDENT_LABEL, 0)), None, None, None]],
                   columns=["band", "n_rows", "lower", "upper", "midpoint_used"]).to_string(index=False))
print("missing salary band:", int(df[COL["salary"]].isna().sum()))

                band  n_rows    lower    upper  midpoint_used
            <$20,000     283      0.0  20000.0        10000.0
   $20,000 - $40,000     318  20000.0  40000.0        30000.0
   $40,000 - $60,000     415  40000.0  60000.0        50000.0
   $60,000 - $80,000     413  60000.0  80000.0        70000.0
  $80,000 - $100,000     292  80000.0 100000.0        90000.0
 $100,000 - $120,000     222 100000.0 120000.0       110000.0
 $120,000 - $140,000      97 120000.0 140000.0       130000.0
           >$140,000     110 140000.0      NaN       150000.0
Student / Unemployed     215      NaN      NaN            NaN
missing salary band: 364


### Cell 3.2 - Drop rows with no usable salary
**Purpose / decision:** drop rows with no band, and (per `CONFIG`) the `Student / Unemployed` band, which is a status, not an amount. Counts logged.

In [15]:
rows_before = len(df)
df = df.loc[df[COL["salary"]].notna()].copy()
log_rows("3.2 salary band blank", rows_before, len(df))
if CONFIG["drop_student_unemployed"]:
    rows_before = len(df)
    df = df.loc[df[COL["salary"]] != STUDENT_LABEL].copy()
    log_rows("3.2 Student / Unemployed band", rows_before, len(df))
log("3.2 salary rows", "rows without a usable salary band removed; " + str(len(df)) + " remain")

[ROWS] 3.2 salary band blank : 2729 -> 2365 (removed 364 )
[ROWS] 3.2 Student / Unemployed band : 2365 -> 2150 (removed 215 )
[LOG] 3.2 salary rows - rows without a usable salary band removed; 2150 remain


## Gate 4 - Multi-answer questions stored as lists

### Cell 4.0 - Helper: option columns to a `;`-separated list
**Purpose:** purchasing involvement, purchase types and products are one column per option in 2011. The later years (2016+) store the same kind of question as one `;`-separated text cell, so 2011 is written the same way. Pooled multi-hot encoding later is then one function for every year.
**Decisions made:** option labels come from row 0; the free-text "other (please specify)" label becomes `Other`; labels are asserted to contain no `;`; a respondent who ticked nothing gets an empty cell (NaN), not an empty string.

In [16]:
def block_to_list(frame, key):
    a, b = BLOCKS[key]
    cols = RAW[a:b]
    labels = [option_labels.iloc[i] for i in range(a, b)]
    nice = ["Other" if l.startswith("other (please specify)") else l for l in labels]
    assert all(";" not in l for l in nice), "an option label contains ';'"
    for col, lab in zip(cols, labels):
        if not lab.startswith("other (please specify)"):
            assert (frame[col].dropna() == lab).all(), "cell text differs from the option label in " + col
    per_row = frame[cols].notna().sum(axis=1)
    print("block:", key, "| cols", a, "to", b - 1, "| options:", nice)
    print("rows by number of options ticked:", per_row.value_counts().sort_index().to_dict())
    def join_row(values):
        picked = [n for n, v in zip(nice, values) if pd.notna(v)]
        return ";".join(picked) if picked else np.nan
    return frame[cols].apply(lambda r: join_row(r.values), axis=1)

### Cell 4.1 - Purchasing involvement, purchase types, products
**Purpose / decision:** convert the three blocks. Nothing is encoded.

In [17]:
purchase_involvement = block_to_list(df, "purchase_involvement")
print()
purchase_types = block_to_list(df, "purchase_types")
print()
tech_products = block_to_list(df, "products")
for name, s in [("PurchaseInvolvement", purchase_involvement), ("PurchaseTypes", purchase_types), ("TechProductsOwned", tech_products)]:
    print(name, "| blank:", int(s.isna().sum()), "| example:", s.dropna().iloc[0])
log("4.1 list blocks", "purchasing involvement, purchase types and products stored as ';' lists")

block: purchase_involvement | cols 8 to 14 | options: ['Influencer', 'Recommender', 'Approver', 'Purchaser', 'Check Writer', 'No Involvement', "I'm a Seller"]
rows by number of options ticked: {1: 1057, 2: 686, 3: 232, 4: 105, 5: 44, 6: 24, 7: 2}

block: purchase_types | cols 15 to 20 | options: ['Hardware', 'Servers', 'Software', 'User Equipment: Monitors, PCs, Laptops', 'Consultants', 'Other']
rows by number of options ticked: {0: 472, 1: 471, 2: 377, 3: 321, 4: 282, 5: 155, 6: 72}

block: products | cols 46 to 62 | options: ['iPhone', 'Android', 'Blackberry', 'Other Smart Phone', 'Regular Mobile Phone', 'Kindle', 'Nook', 'Blu-Ray', 'HDTV', 'AppleTV', 'iPad', 'Other netbook', 'PS3', 'Xbox', 'Wii', 'Other gaming system', 'Other']
rows by number of options ticked: {0: 60, 1: 357, 2: 452, 3: 402, 4: 308, 5: 258, 6: 153, 7: 84, 8: 55, 9: 13, 10: 5, 11: 1, 12: 1, 14: 1}
PurchaseInvolvement | blank: 0 | example: Influencer;Approver;Purchaser
PurchaseTypes | blank: 472 | example: Hardware;S

### Cell 4.2 - Budget for outside expenditures: this is single-answer with some conflicts
**Purpose:** 99% of answering rows pick exactly one bracket. The conflicting rows are resolved with your rules, then the block is stored as one text column.
**Decisions made:** (1) `Don't know` wins; (2) otherwise `>$150,000` wins; (3) otherwise (my addition, review me) the highest bracket ticked is kept.

In [18]:
a, b = BLOCKS["budget"]
budget_cols = RAW[a:b]
budget_labels = [option_labels.iloc[i] for i in range(a, b)]
for col, lab in zip(budget_cols, budget_labels):
    assert (df[col].dropna() == lab).all()
selected = df[budget_cols].notna()
selected.columns = budget_labels
multi_mask = selected.sum(axis=1) > 1
print("rows with more than one bracket ticked:", int(multi_mask.sum()))

resolved = selected.copy()
ordered_money = [l for l in budget_labels if l != "Don't know"]
rule1 = multi_mask & selected["Don't know"]
resolved.loc[rule1, :] = False
resolved.loc[rule1, "Don't know"] = True
rule2 = multi_mask & ~rule1 & selected[">$150,000"]
resolved.loc[rule2, :] = False
resolved.loc[rule2, ">$150,000"] = True
rule3 = multi_mask & ~rule1 & ~rule2
for idx in resolved.index[rule3]:
    keep = ordered_money[max(k for k, l in enumerate(ordered_money) if selected.loc[idx, l])]
    resolved.loc[idx, :] = False
    resolved.loc[idx, keep] = True
print("rule 1:", int(rule1.sum()), "| rule 2:", int(rule2.sum()), "| rule 3:", int(rule3.sum()))
assert resolved.sum(axis=1).max() <= 1

budget = pd.Series(np.nan, index=df.index, dtype=object)
has_answer = resolved.sum(axis=1) == 1
budget[has_answer] = resolved[has_answer].idxmax(axis=1)
print(budget.value_counts(dropna=False).to_string())
log("4.2 budget", str(int(multi_mask.sum())) + " conflicting rows resolved (rule1 " + str(int(rule1.sum())) + ", rule2 " + str(int(rule2.sum())) + ", rule3 " + str(int(rule3.sum())) + "); stored as one text column")

rows with more than one bracket ticked: 26
rule 1: 11 | rule 2: 1 | rule 3: 14
Don't know             795
NaN                    472
<$10,000               408
$10,001 - $25,000      188
>$150,000              111
$25,001 - $40,000       72
$100,001 - $150,000     38
$41,000 - $75,000       34
$75,001 - $100,000      32
[LOG] 4.2 budget - 26 conflicting rows resolved (rule1 11, rule2 1, rule3 14); stored as one text column


## Gate 5 - Language proficiency (cols 30-42)

### Cell 5.1 - Look at the block
**Purpose:** 12 tick-boxes plus one free-text box. Print the labels and the write-in volume before deciding anything.
**Decision made:** none.

In [19]:
a, b = BLOCKS["language"]
lang_cols = RAW[a:b]
lang_labels = [option_labels.iloc[i] for i in range(a, b)]
for i, lab in zip(range(a, b), lang_labels):
    print("col", i, "->", lab, "| ticked:", int(df[RAW[i]].notna().sum()))
WRITEIN_COL = lang_cols[-1]
assert lang_labels[-1].startswith("other (please specify)")
writein = df[WRITEIN_COL].dropna()
print()
print("write-in answers:", len(writein), "| distinct as typed:", writein.nunique(), "| distinct lowercased:", writein.str.lower().str.strip().nunique())
print("most common (lowercased):")
print(writein.str.lower().str.strip().value_counts().head(15).to_string())

col 30 -> Java | ticked: 708
col 31 -> JavaScript | ticked: 1203
col 32 -> CSS | ticked: 1133
col 33 -> PHP | ticked: 638
col 34 -> Python | ticked: 458
col 35 -> Ruby | ticked: 227
col 36 -> SQL | ticked: 1396
col 37 -> C# | ticked: 1156
col 38 -> C++ | ticked: 580
col 39 -> C | ticked: 593
col 40 -> Perl | ticked: 251
col 41 -> None | ticked: 13
col 42 -> other (please specify) | ticked: 533

write-in answers: 533 | distinct as typed: 248 | distinct lowercased: 208
most common (lowercased):
Unnamed: 42
objective-c     52
vb.net          42
delphi          42
java            26
vb              22
coldfusion      15
objective c     15
scala           12
actionscript     9
bash             8
lua              8
html             8
f#               8
fortran          7
groovy           6


### Cell 5.2 - What the 12 tick-boxes become (canonical names)
**Purpose:** a tick-box and a write-in must end up with the same name. This table maps each tick-box to its canonical language name.
**Decisions made (review me):**
- `CSS` becomes **`HTML/CSS`**, because that is how later years name it, and write-ins that say `html` or `xhtml` join it.
- `None` means no languages: it adds nothing to the list and the count is 0 (not missing).
- Canonical names follow the 2021+ style (`Bash/Shell`, `HTML/CSS`, `Objective-C`). The mapping lives in this one table, so renaming to your shared vocabulary is a one-line change.

In [20]:
CHECKBOX_MAP = {"Java": "Java", "JavaScript": "JavaScript", "CSS": "HTML/CSS", "PHP": "PHP", "Python": "Python",
                "Ruby": "Ruby", "SQL": "SQL", "C#": "C#", "C++": "C++", "C": "C", "Perl": "Perl", "None": None}
print(pd.DataFrame([[k, v] for k, v in CHECKBOX_MAP.items()], columns=["tick-box label (row 0)", "canonical name"]).to_string(index=False))
assert set(lang_labels[:-1]) == set(CHECKBOX_MAP), sorted(set(lang_labels[:-1]) ^ set(CHECKBOX_MAP))

tick-box label (row 0) canonical name
                  Java           Java
            JavaScript     JavaScript
                   CSS       HTML/CSS
                   PHP            PHP
                Python         Python
                  Ruby           Ruby
                   SQL            SQL
                    C#             C#
                   C++            C++
                     C              C
                  Perl           Perl
                  None            NaN


### Cell 5.3 - Alias dictionary for the write-ins
**Purpose:** turn free text into canonical names. People typed several languages per box, in mixed case, with typos, abbreviations (`vb`, `obj-c`, `as3`) and commentary.
**Method:** each rule is a pattern searched anywhere in the lowercased text, and a match is *removed* before the next rule runs, so `vb.net` is not also counted as `vb`. More specific rules come first.
**Decisions made (review me):**
- **Languages are counted; tools are not.** Frameworks, platforms and markup (jQuery, ASP.NET, Flash/Flex, XML, LaTeX, Android...) are recognised but kept out of the language list and count.
- **Visual Basic stays split three ways:** `VB.NET`, `Visual Basic 6` (`vb6`, `classic vb`) and plain `Visual Basic` (`vb`, which does not say which). Merge them later if you wish; I did not guess.
- `Object Pascal` -> `Delphi`; `PL/SQL` stays separate from `SQL`; `Scheme` stays separate from `Lisp`.
- One-letter names (`R`, `D`, `J`, `Go`) only count when that is the whole phrase, because a single `r` inside a sentence is not a language.
- Anything left over is kept as `unrecognized` for audit and never counted.

In [21]:
def _name_regex(name):
    return r"\b" + re.escape(name.lower()).replace(r"\ ", r"\s+") + r"\b"

LANG_RULES = [
    ("VB.NET", r"visual\s*basic\s*\.?\s*net|\bvb\s*\.?\s*net"),
    ("VBScript", r"\bvb\s*script|\bvbs\b"),
    ("VBA", r"\bvba\b"),
    ("Visual Basic 6", r"\bvb\s*6\b|visual\s*basic\s*6|classic\s*vb"),
    ("Visual Basic", r"visual\s*basic|\bvb\b"),
    ("PL/SQL", r"\bpl\s*/\s*sql"),
    ("SQL", r"\bsql\b"),
    ("Objective-C", r"\bobj(?:ective|ectve)?[\s\-]*c\b"),
    ("C#", r"\bc\s?#"),
    ("C++", r"\bc\s?\+\+|\bcpp\b"),
    ("F#", r"\bf\s?#"),
    ("Java", r"\bjava\b"),
    ("JavaScript", r"\bjavascript\b|\becmascript\b"),
    ("PHP", r"\bphp\b"),
    ("Python", r"\bpython\b"),
    ("Ruby", r"\bruby\b"),
    ("Perl", r"\bperl\b"),
    ("HTML/CSS", r"\bx?html5?\b|\bcss3?\b"),
    ("PowerShell", r"powershell"),
    ("Bash/Shell", r"\bbash\b|\bshell?(?:\s+script\w*)?\b|\bksh\b|\bzsh\b|\bcsh\b|\bsh\b"),
    ("Assembly", r"(?:x86\s+|arm\s+|z/)?assembl(?:y|er)\b(?:\s+language)?"),
    ("Delphi", r"delphi|object\s*pascal|\boo-pascal\b"),
    ("Pascal", r"\bpascal\b"),
    ("ActionScript", r"actionscrip\w*(?:\s*3(?:\.0)?)?|\bas[123](?:-[123])?\b"),
    ("ColdFusion", r"coldfusion|\bcfml\b"),
    ("Lisp", r"\blisp\b"),
    ("MEL", r"\bmel\b"),
]
RARE_LANGS = ["Scheme", "Clojure", "Haskell", "Scala", "Erlang", "Lua", "Groovy", "OCaml", "Prolog", "Fortran", "COBOL", "Ada",
              "Tcl", "AWK", "Smalltalk", "Eiffel", "Forth", "Rexx", "MATLAB", "Mathematica", "Maple", "IDL", "SAS", "RPG", "ABAP",
              "PowerBuilder", "FoxPro", "LabVIEW", "AppleScript", "VHDL", "Verilog", "AutoHotkey", "AutoIt", "LSL", "Avisynth",
              "KRL", "Wasabi", "GeneXus", "LotusScript", "UniBasic", "UniQuery", "Nemerle", "Gosu", "Vala", "Brainfuck", "Befunge",
              "LOLCODE", "RealBasic", "CCScript", "DataFlex", "WebFocus", "Linc 4GL", "Forte 4GL", "Progress ABL",
              "JMP Scripting Language", "XSLT", "Lansa", "BASIC"]
LANG_RULES += [(n, _name_regex(n)) for n in RARE_LANGS]

TOOL_RULES = [
    ("ASP.NET", r"\basp\.net(?:-mvc)?"), ("ASP", r"\basp\b(?:\s*(?:classic|3\.0))?"), (".NET", r"(?<!\w)\.net\b"),
    ("jQuery", r"\bjquery\b"), ("JSP", r"\bjsp\b"), ("Symfony", r"\bsymfony\b"), ("Silverlight", r"\bsilverlight\b"),
    ("WPF", r"\bwpf\b"), ("WCF", r"\bwcf\b"), ("Entity Framework", r"entity-framework"), ("Grails", r"\bgrails\b"),
    ("Cocoa", r"\bcocoa\b"), ("Android", r"\bandroid\b"), ("Flex", r"(?:adobe\s+)?\bflex\b"), ("Flash", r"\bflash\b"),
    ("Excel", r"\bexcel\b"), ("NumPy", r"\bnumpy\b"), ("LaTeX", r"\blatex\b"), ("XML", r"\bxml\b"), ("XAML", r"\bxaml\b"),
    ("X3D", r"\bx3d\b"), ("XPath", r"\bxpath\b"), ("sed", r"\bsed\b"),
]
EXACT_LANG = {"r": "R", "d": "D", "j": "J", "go": "Go"}
EXACT_TOOL = {"make": "Make"}
# two answers where commentary contains language-like words; decided by hand and shown in Cell 5.5
MANUAL = {"css is not a programming language.": set(),
          "vb (why the fuck was this not a choice? c# snobs?)": {"Visual Basic"}}

COMPILED_LANG = [(n, re.compile(p)) for n, p in LANG_RULES]
COMPILED_TOOL = [(n, re.compile(p)) for n, p in TOOL_RULES]

def parse_writein(text):
    s = str(text).lower().strip().replace("’", "'")
    if s in MANUAL:
        return set(MANUAL[s]), set(), ["(manual override)"]
    langs, tools = set(), set()
    for name, rx in COMPILED_LANG:
        if rx.search(s):
            langs.add(name)
            s = rx.sub(" | ", s)
    for name, rx in COMPILED_TOOL:
        if rx.search(s):
            tools.add(name)
            s = rx.sub(" | ", s)
    leftovers = []
    for phrase in re.split(r"[|,;/&+()\[\]:!?]+", s):
        p = re.sub(r"[^a-z0-9#+.\- ]", "", phrase).strip(" .-")
        if not p:
            continue
        if p in EXACT_LANG:
            langs.add(EXACT_LANG[p])
        elif p in EXACT_TOOL:
            tools.add(EXACT_TOOL[p])
        else:
            leftovers.append(p)
    return langs, tools, leftovers

alias_table = pd.DataFrame([[n, "language", p] for n, p in LANG_RULES] + [[n, "tool/markup (not counted)", p] for n, p in TOOL_RULES] +
                           [[v, "language (exact phrase only)", "^" + k + "$"] for k, v in EXACT_LANG.items()],
                           columns=["canonical_name", "kind", "pattern"])
print("alias rules:", len(alias_table), "| language names:", int((alias_table.kind == "language").sum()), "| tool names:", int(alias_table.kind.str.startswith("tool").sum()))
for test in ["vb.net, vb6", "objectve c", "Objective-C (how dare you leave that out)! :-p", "r vb", "x86 assembler oo-pascal", "css is not a programming language."]:
    print(repr(test), "->", parse_writein(test))

alias rules: 112 | language names: 85 | tool names: 23
'vb.net, vb6' -> ({'VB.NET', 'Visual Basic 6'}, set(), [])
'objectve c' -> ({'Objective-C'}, set(), [])
'Objective-C (how dare you leave that out)! :-p' -> ({'Objective-C'}, set(), ['how dare you leave that out', 'p'])
'r vb' -> ({'Visual Basic', 'R'}, set(), [])
'x86 assembler oo-pascal' -> ({'Delphi', 'Assembly'}, set(), [])
'css is not a programming language.' -> (set(), set(), ['(manual override)'])


### Cell 5.4 - Parse every write-in and show the result
**Purpose:** apply the dictionary to all write-ins in the retained rows. The table shows, for the canonical names found, how many respondents wrote each one.
**Decision made:** none; this is the audit step.

In [22]:
parsed = {}
for idx, text in writein.items():
    parsed[idx] = parse_writein(text)

lang_counts = {}
for idx, (langs, tools, left) in parsed.items():
    for name in langs:
        lang_counts[name] = lang_counts.get(name, 0) + 1
print(pd.Series(lang_counts).sort_values(ascending=False).to_frame("respondents").T.to_string())
print()
n_none = sum(1 for v in parsed.values() if not v[0])
print("write-ins with no language recognised:", n_none, "of", len(parsed))

             Objective-C  VB.NET  Delphi  Visual Basic  Java  ActionScript  Bash/Shell  ColdFusion  HTML/CSS  Scala  VBA  F#  Lisp  Assembly  Visual Basic 6  Groovy  Lua  Haskell  MATLAB  Fortran  PowerShell  R  XSLT  Tcl  VBScript  Prolog  LabVIEW  Scheme  OCaml  Ada  PL/SQL  Clojure  RPG  AWK  IDL  Rexx  PowerBuilder  COBOL  BASIC  Erlang  ABAP  Smalltalk  FoxPro  SAS  JavaScript  Maple  Mathematica  DataFlex  AppleScript  Progress ABL  Ruby  LotusScript  SQL  MEL  C#  Linc 4GL  WebFocus  Forte 4GL  Go  UniQuery  D  VHDL  CCScript  Brainfuck  Wasabi  JMP Scripting Language  UniBasic  Befunge  AutoIt  Lansa  Gosu  J  Vala  Pascal  RealBasic
respondents           83      60      50            40    39            24          20          19        16     15   12  12    12        11              11      10   10        9       8        8           8  7     6    6         6       6        5       5      4    4       4        4    3    3    2     2             2      2      2       2     2  

### Cell 5.5 - Review the odd ones
**Purpose:** two checks that the parser did not invent languages. (1) every write-in longer than 25 characters or containing punctuation that signals commentary, with its parse; (2) the most common leftover phrases, in case a real language was missed.
**Decision made:** none. Any miss is fixed by adding a rule in Cell 5.3, not by editing data.

In [23]:
long_ones = [(idx, t) for idx, t in writein.items() if len(t) > 25 or "?" in t or "!" in t]
print("commentary-style write-ins:", len(long_ones))
for idx, t in long_ones:
    langs, tools, left = parsed[idx]
    print("row", idx, "|", repr(t[:90]))
    print("      languages:", sorted(langs), "| tools:", sorted(tools), "| leftover:", left[:4])
print()
leftover_counts = {}
for langs, tools, left in parsed.values():
    for p in left:
        leftover_counts[p] = leftover_counts.get(p, 0) + 1
print("most common leftover phrases:")
print(pd.Series(leftover_counts).sort_values(ascending=False).head(25).to_string())

commentary-style write-ins: 19
row 143 | 'If CSS is a language, surely so is HTML?'
      languages: ['HTML/CSS'] | tools: [] | leftover: ['if', 'is a language', 'surely so is']
row 423 | '"Proficient" seems too rosey a word for it.'
      languages: [] | tools: [] | leftover: ['proficient seems too rosey a word for it']
row 511 | 'Actionscript 3, Objective-C'
      languages: ['ActionScript', 'Objective-C'] | tools: [] | leftover: []
row 1045 | 'Objective-C (how dare you leave that out)! :-P'
      languages: ['Objective-C'] | tools: [] | leftover: ['how dare you leave that out', 'p']
row 1158 | 'VB.NET!'
      languages: ['VB.NET'] | tools: [] | leftover: []
row 1364 | 'Common Lisp, Clojure, Scala, LaTeX'
      languages: ['Clojure', 'Lisp', 'Scala'] | tools: ['LaTeX'] | leftover: ['common']
row 1502 | 'vb. technologies: asp.net, wpf, wcf'
      languages: ['Visual Basic'] | tools: ['ASP.NET', 'WCF', 'WPF'] | leftover: ['technologies']
row 1512 | 'JavaScript (incl. jQuery), HTML'
   

### Cell 5.6 - Build `LanguageHaveWorkedWith` and the count
**Purpose:** combine the tick-boxes and the parsed write-ins into one canonical list per respondent, plus a count.
**Decisions made:**
- The list is the **distinct** canonical names, sorted alphabetically, joined with `;`.
- `LanguageHaveWorkedWith_count` is the length of that list. It is `0` only for people who ticked `None` with no language. It is empty (NaN, meaning unknown) for people who left the block blank **and** for people whose only answer was a write-in with no recognisable language (for example "various"): they did not say they know no languages. Someone who ticked `None` and also named a language keeps the language.
- Names that are in both the tick-boxes and the write-in are counted once.
- Caution for the pooled model: the 2011 question offered 12 boxes, while 2021+ offer 40+, so a count of 4 does not mean the same in every year. Compare counts within a year, or let the model see `survey_year`.

In [24]:
tick_cols = lang_cols[:-1]
tick_labels = lang_labels[:-1]
final_lists, final_counts, source_flags = [], [], []
for idx in df.index:
    names = set()
    none_ticked = False
    for col, lab in zip(tick_cols, tick_labels):
        if pd.notna(df.at[idx, col]):
            if lab == "None":
                none_ticked = True
            else:
                names.add(CHECKBOX_MAP[lab])
    if idx in parsed:
        names |= parsed[idx][0]
    answered = bool(df.loc[idx, lang_cols].notna().any())
    final_lists.append(";".join(sorted(names, key=str.lower)) if names else np.nan)
    if names:
        final_counts.append(len(names))
    elif none_ticked:
        final_counts.append(0)
    else:
        final_counts.append(np.nan)   # block blank, or only a write-in with no recognisable language
    source_flags.append(none_ticked)

language_list = pd.Series(final_lists, index=df.index, dtype=object)
language_count = pd.Series(final_counts, index=df.index, dtype=float)
none_ticked = pd.Series(source_flags, index=df.index)
block_answered = df[lang_cols].notna().any(axis=1)
print("rows:", len(df), "| count blank:", int(language_count.isna().sum()),
      "(block unanswered:", int((~block_answered).sum()), "| answered but only an unrecognisable write-in:", int((block_answered & language_list.isna() & ~none_ticked).sum()), ")")
print("'None' ticked:", int(none_ticked.sum()), "| of those, rows that also have a language:", int((none_ticked & language_list.notna()).sum()), "| count = 0 rows:", int((language_count == 0).sum()))
print("count distribution:")
print(language_count.value_counts().sort_index().to_string())
print("mean count: %.2f" % language_count.mean())
log("5.6 languages", "tick-boxes + parsed write-ins -> LanguageHaveWorkedWith list and count")

rows: 2150 | count blank: 3 (block unanswered: 1 | answered but only an unrecognisable write-in: 2 )
'None' ticked: 13 | of those, rows that also have a language: 2 | count = 0 rows: 11
count distribution:
0.0      11
1.0     151
2.0     263
3.0     383
4.0     502
5.0     387
6.0     211
7.0     120
8.0      61
9.0      39
10.0     13
11.0      3
12.0      1
13.0      1
16.0      1
mean count: 4.15
[LOG] 5.6 languages - tick-boxes + parsed write-ins -> LanguageHaveWorkedWith list and count


### Cell 5.7 - Sanity checks and audit file
**Purpose:** confirm no name is spelled two ways, the list round-trips to the count, and write out an audit file so every parsed write-in can be checked by eye.
**Decision made:** none.

In [25]:
all_names = sorted({n for s in language_list.dropna() for n in s.split(";")}, key=str.lower)
print("distinct canonical language names in the column:", len(all_names))
print(all_names)
lowered = {}
for n in all_names:
    lowered.setdefault(n.lower(), []).append(n)
assert all(len(v) == 1 for v in lowered.values()), "two spellings of the same name"
for idx in df.index[:2000]:
    s = language_list[idx]
    n = 0 if pd.isna(s) else len(s.split(";"))
    c = language_count[idx]
    assert (pd.isna(c) and pd.isna(s)) or c == n, "count does not match list at row " + str(idx)
print("count matches list length for every row")

audit = pd.DataFrame({"source_row": list(parsed.keys()), "as_typed": [writein[i] for i in parsed],
                      "languages_found": [";".join(sorted(parsed[i][0])) for i in parsed],
                      "tools_not_counted": [";".join(sorted(parsed[i][1])) for i in parsed],
                      "unrecognized_text": [" | ".join(parsed[i][2]) for i in parsed]})
audit.to_csv("lang_writein_audit_2011.csv", index=False)
alias_table.to_csv("language_alias_table.csv", index=False)
print("saved lang_writein_audit_2011.csv (%d rows) and language_alias_table.csv (%d rules)" % (len(audit), len(alias_table)))

distinct canonical language names in the column: 80
['ABAP', 'ActionScript', 'Ada', 'AppleScript', 'Assembly', 'AutoIt', 'AWK', 'Bash/Shell', 'BASIC', 'Befunge', 'Brainfuck', 'C', 'C#', 'C++', 'CCScript', 'Clojure', 'COBOL', 'ColdFusion', 'D', 'DataFlex', 'Delphi', 'Erlang', 'F#', 'Forte 4GL', 'Fortran', 'FoxPro', 'Go', 'Gosu', 'Groovy', 'Haskell', 'HTML/CSS', 'IDL', 'J', 'Java', 'JavaScript', 'JMP Scripting Language', 'LabVIEW', 'Lansa', 'Linc 4GL', 'Lisp', 'LotusScript', 'Lua', 'Maple', 'Mathematica', 'MATLAB', 'MEL', 'Objective-C', 'OCaml', 'Pascal', 'Perl', 'PHP', 'PL/SQL', 'PowerBuilder', 'PowerShell', 'Progress ABL', 'Prolog', 'Python', 'R', 'RealBasic', 'Rexx', 'RPG', 'Ruby', 'SAS', 'Scala', 'Scheme', 'Smalltalk', 'SQL', 'Tcl', 'UniBasic', 'UniQuery', 'Vala', 'VB.NET', 'VBA', 'VBScript', 'VHDL', 'Visual Basic', 'Visual Basic 6', 'Wasabi', 'WebFocus', 'XSLT']
count matches list length for every row
saved lang_writein_audit_2011.csv (533 rows) and language_alias_table.csv (112 rul

## Gate 6 - Single-answer questions (kept as they are)

### Cell 6.1 - One repair: Excel damaged the experience bands
**Purpose:** the experience column contains `41310`, `41435` and `11` instead of band labels. These are Excel date serials (decoded below: 5 Feb 2013 and 10 Jun 2013, i.e. `2-5` and `6-10` read as day-month), and `11` is `11+` with the plus lost.
**Decision made:** repair only this column, back to the labels the survey used. This is fixing corruption, not encoding. Every other single-answer column keeps its text exactly as the respondent chose it.

In [26]:
for serial in (41310, 41435):
    print(serial, "->", datetime.date(1899, 12, 30) + datetime.timedelta(days=serial))
EXP_FIX = {"<2": "<2", "41310": "2-5", "41435": "6-10", "11": "11+"}
print("before:", df[COL["exp"]].value_counts(dropna=False).to_dict())
unmapped = sorted(set(df[COL["exp"]].dropna()) - set(EXP_FIX))
assert not unmapped, unmapped
years_code = df[COL["exp"]].map(EXP_FIX)
print("after: ", years_code.value_counts(dropna=False).to_dict())
log("6.1 YearsCode", "Excel serials 41310->2-5, 41435->6-10 and bare 11->11+ repaired; no other single-answer column changed")

41310 -> 2013-02-05
41435 -> 2013-06-10
before: {'11': 896, '41435': 662, '41310': 518, '<2': 74}
after:  {'11+': 896, '6-10': 662, '2-5': 518, '<2': 74}
[LOG] 6.1 YearsCode - Excel serials 41310->2-5, 41435->6-10 and bare 11->11+ repaired; no other single-answer column changed


### Cell 6.2 - Value sets of the single-answer columns
**Purpose:** print every value set so you can see exactly what stays. **Decision made:** none; no merging, no imputation, blanks stay empty.

In [27]:
for k in ["country", "state", "age", "industry", "size", "occ", "rec", "project", "os", "sat", "spend", "site"]:
    s = df[COL[k]]
    print("---", k, "|", COL[k][:70], "| distinct:", s.nunique(), "| blank:", int(s.isna().sum()))
    if s.nunique() <= 15:
        print(s.value_counts(dropna=False).to_string())

--- country | What Country or Region do you live in? | distinct: 19 | blank: 0
--- state | Which US State or Territory do you live in? | distinct: 50 | blank: 1312
--- age | How old are you? | distinct: 8 | blank: 0
How old are you?
25-29    691
30-34    536
20-24    347
35-39    301
40-50    230
51-60     29
< 20      13
>60        3
--- industry | How would you best describe the industry you work in? | distinct: 12 | blank: 0
How would you best describe the industry you work in?
Software Products          661
Web Services               392
Consulting                 274
Other                      270
Finance / Banking          168
Education                   91
Manufacturing               70
Healthcare                  68
Retail                      49
Gaming                      38
Advertising                 37
Foundation / Non-Profit     32
--- size | Which best describes the size of your company? | distinct: 6 | blank: 0
Which best describes the size of your company?
Start Up (1-

## Gate 7 - Headers and assembly

### Cell 7.1 - Header map
**Purpose:** rename every column to the shared name, and record the original header, status and a note.
**Decisions made (review me):** the **confirmed** names are the ones in your shared list (`Country`, `Age`, `YearsCode`, `OrgSize`, `DevType`, `LanguageHaveWorkedWith`, `OpSys`, `JobSat`, `CompTotal`, `ConvertedCompYearly`). I could not see your Canonical Mapping spreadsheet, so every other name is **proposed**: check it against the sheet and change it here in one place. `ResponseId` is the row number in the 2011 file, since 2011 has no respondent id.

In [28]:
HEADER_MAP = [
    # canonical name, status, original header / span, note
    ("ResponseId", "proposed", "(none in 2011)", "row number of the respondent in the 2011 file"),
    ("survey_year", "proposed", "(added)", "constant 2011"),
    ("Country", "confirmed", COL["country"], "2011 values are regions as well as countries (e.g. 'Other Europe'), not comparable with later country names"),
    ("USState", "proposed", COL["state"], "only asked of US respondents"),
    ("Age", "confirmed", COL["age"], "text bands, kept as typed"),
    ("YearsCode", "confirmed", COL["exp"], "text bands '<2','2-5','6-10','11+' after Excel repair; question says IT/programming experience - check vs YearsCode/YearsCodePro"),
    ("Industry", "proposed", COL["industry"], ""),
    ("OrgSize", "confirmed", COL["size"], "text bands incl. 'Student' and 'Other (not working...)'"),
    ("DevType", "confirmed", COL["occ"], "single answer in 2011; later years may be ';' lists"),
    ("RecommendationActedUpon", "proposed", COL["rec"], "may relate to PurchaseInfluence in later years - check"),
    ("PurchaseInvolvement", "proposed", RAW[8] + " (cols 8-14)", "';' list of options ticked"),
    ("PurchaseTypes", "proposed", RAW[15] + " (cols 15-20)", "';' list"),
    ("BudgetOutsideExpenditure", "proposed", RAW[21] + " (cols 21-28)", "single text bracket after conflict rules; 'Don't know' kept as a value"),
    ("ProjectType", "proposed", COL["project"], ""),
    ("LanguageHaveWorkedWith", "confirmed", RAW[30] + " (cols 30-42)", "';' list of canonical names; tick-boxes + parsed write-ins; 2011 wording is 'proficient in'"),
    ("LanguageHaveWorkedWith_count", "proposed", "(derived)", "number of distinct languages; 0 if 'None' ticked; blank if block unanswered"),
    ("OpSys", "confirmed", COL["os"], "text as typed (Windows 7 / XP / Vista kept separate)"),
    ("JobSat", "confirmed", COL["sat"], "five text answers in 2011; later years use a 0-10 score, harmonise later"),
    ("CompTotal", "confirmed", COL["salary"], "salary BAND text, kept as typed"),
    ("ConvertedCompYearly", "confirmed", "(derived from " + COL["salary"][:40] + "...)", "annual USD band midpoint; includes bonus; top band set by CONFIG"),
    ("TechProductsOwned", "proposed", RAW[46] + " (cols 46-62)", "';' list"),
    ("PersonalTechSpend12m", "proposed", COL["spend"], "text bands, kept as typed"),
    ("SitesFrequentedMost", "proposed", COL["site"], "text as typed; includes a real answer 'None' - load the CSV with keep_default_na=False, na_values=['']"),
]
header_map = pd.DataFrame(HEADER_MAP, columns=["canonical_name", "status", "original_header_or_span", "note"])
print(header_map.status.value_counts().to_dict())
print(header_map[["canonical_name", "status", "note"]].to_string(index=False, max_colwidth=90))

{'proposed': 13, 'confirmed': 10}
              canonical_name    status                                                                                       note
                  ResponseId  proposed                                              row number of the respondent in the 2011 file
                 survey_year  proposed                                                                              constant 2011
                     Country confirmed 2011 values are regions as well as countries (e.g. 'Other Europe'), not comparable with...
                     USState  proposed                                                               only asked of US respondents
                         Age confirmed                                                                  text bands, kept as typed
                   YearsCode confirmed text bands '<2','2-5','6-10','11+' after Excel repair; question says IT/programming exp...
                    Industry  proposed                  

### Cell 7.2 - Assemble the canonical table
**Purpose:** put the columns together in the original questionnaire order, with the shared names.
**Decision made:** none new. Blanks stay blank; nothing is imputed, merged or encoded.

In [29]:
out = pd.DataFrame(index=df.index)
out["ResponseId"] = df.index.astype(int)
out["survey_year"] = 2011
out["Country"] = df[COL["country"]]
out["USState"] = df[COL["state"]]
out["Age"] = df[COL["age"]]
out["YearsCode"] = years_code
out["Industry"] = df[COL["industry"]]
out["OrgSize"] = df[COL["size"]]
out["DevType"] = df[COL["occ"]]
out["RecommendationActedUpon"] = df[COL["rec"]]
out["PurchaseInvolvement"] = purchase_involvement
out["PurchaseTypes"] = purchase_types
out["BudgetOutsideExpenditure"] = budget
out["ProjectType"] = df[COL["project"]]
out["LanguageHaveWorkedWith"] = language_list
out["LanguageHaveWorkedWith_count"] = language_count
out["OpSys"] = df[COL["os"]]
out["JobSat"] = df[COL["sat"]]
out["CompTotal"] = df[COL["salary"]]
label_to_mid = {b[0]: b[3] for b in BANDS}
out["ConvertedCompYearly"] = df[COL["salary"]].map(label_to_mid).astype(float)
out["TechProductsOwned"] = tech_products
out["PersonalTechSpend12m"] = df[COL["spend"]]
out["SitesFrequentedMost"] = df[COL["site"]]

assert list(out.columns) == list(header_map.canonical_name), "header map and table columns differ"
print("shape:", out.shape)
print(out.head(3).T.to_string(max_colwidth=60))

shape: (2150, 23)
                                                                                        5                                                            7                                                   8
ResponseId                                                                              5                                                            7                                                   8
survey_year                                                                          2011                                                         2011                                                2011
Country                                                                        Other Asia                                     United States of America                            United States of America
USState                                                                               NaN                                                     Illinois                    

### Cell 7.3 - Validate
**Purpose:** prove the file is what it claims: right row count, no encoded columns, no empty strings, list columns well-formed, text answers unchanged from the file, missing values left as missing.
**Decision made:** none.

In [30]:
assert len(out) == len(df)
list_cols = ["PurchaseInvolvement", "PurchaseTypes", "LanguageHaveWorkedWith", "TechProductsOwned"]
for c in list_cols:
    s = out[c].dropna()
    assert not s.str.startswith(";").any() and not s.str.endswith(";").any() and not s.str.contains(";;").any(), c
    assert all(len(v.split(";")) == len(set(v.split(";"))) for v in s), "duplicate item in " + c
    print(c, "| blank:", int(out[c].isna().sum()), "| mean items: %.2f" % s.str.split(";").str.len().mean())
text_cols = [c for c in out.columns if not pd.api.types.is_numeric_dtype(out[c])]
for c in text_cols:
    assert not (out[c].dropna().str.strip() == "").any(), "empty string in " + c
numeric_cols = [c for c in out.columns if c not in text_cols]
na_like = {"None", "N/A", "NA", "n/a", "null", "NULL", "NaN", "nan"}
hits = {c: int(out[c].isin(na_like).sum()) for c in text_cols}
print("cells holding NA-like TEXT that is a real answer (read the CSV with keep_default_na=False):", {k: v for k, v in hits.items() if v})
print("numeric columns:", numeric_cols)
print("text columns:", len(text_cols))

for k, c in [("country", "Country"), ("age", "Age"), ("size", "OrgSize"), ("occ", "DevType"), ("os", "OpSys"), ("sat", "JobSat"), ("salary", "CompTotal")]:
    assert out[c].equals(df[COL[k]]), c + " differs from the file"
print("single-answer columns equal the file exactly (Country, Age, OrgSize, DevType, OpSys, JobSat, CompTotal)")
print("missing per column:")
print(out.isna().sum()[out.isna().sum() > 0].to_string())
print("any one-hot style 0/1 columns:", [c for c in numeric_cols if set(out[c].dropna().unique()) <= {0, 1} and c != "survey_year"])

PurchaseInvolvement | blank: 0 | mean items: 1.82
PurchaseTypes | blank: 472 | mean items: 2.70
LanguageHaveWorkedWith | blank: 14 | mean items: 4.18
TechProductsOwned | blank: 60 | mean items: 3.42
cells holding NA-like TEXT that is a real answer (read the CSV with keep_default_na=False): {'SitesFrequentedMost': 9}
numeric columns: ['ResponseId', 'survey_year', 'LanguageHaveWorkedWith_count', 'ConvertedCompYearly']
text columns: 19
single-answer columns equal the file exactly (Country, Age, OrgSize, DevType, OpSys, JobSat, CompTotal)
missing per column:
USState                         1312
RecommendationActedUpon           11
PurchaseTypes                    472
BudgetOutsideExpenditure         472
LanguageHaveWorkedWith            14
LanguageHaveWorkedWith_count       3
OpSys                              5
TechProductsOwned                 60
PersonalTechSpend12m              11
SitesFrequentedMost               15
any one-hot style 0/1 columns: []


### Cell 7.4 - Save
**Purpose:** write `canonical_2011.csv` and `header_map_2011.csv`, then read the CSV back and compare.
**Decision made:** none.

In [31]:
out.to_csv("canonical_2011.csv", index=False)
header_map.to_csv("header_map_2011.csv", index=False)
back = pd.read_csv("canonical_2011.csv", keep_default_na=False, na_values=[""])
print("saved canonical_2011.csv | on disk:", back.shape, "| in memory:", out.shape)
assert back.shape == out.shape
assert list(back.columns) == list(out.columns)
assert (back.isna().values == out.isna().values).all(), "missing-value pattern changed on save"
for c in out.columns:
    if not pd.api.types.is_numeric_dtype(out[c]):
        assert (back[c].fillna("").values == out[c].fillna("").values).all(), "text changed on save in " + c
    else:
        assert np.allclose(back[c].fillna(-1).values, out[c].fillna(-1).values), "numbers changed on save in " + c
print("round trip identical: same shape, same missing pattern, same text and numbers in all", out.shape[1], "columns")
log("7.4 save", "canonical_2011.csv: " + str(out.shape[0]) + " rows x " + str(out.shape[1]) + " columns, not encoded")
print()
print("row log:")
for step, b, a in row_log:
    print("  %-48s %5d -> %5d" % (step, b, a))
print()
print("decision log:")
for step, text in decision_log:
    print(" ", step, "|", text)

saved canonical_2011.csv | on disk: (2150, 23) | in memory: (2150, 23)
round trip identical: same shape, same missing pattern, same text and numbers in all 23 columns
[LOG] 7.4 save - canonical_2011.csv: 2150 rows x 23 columns, not encoded

row log:
  1.6 option-label row moved out of data            2814 ->  2813
  2.2 junk rows (only Country/State answered)       2813 ->  2729
  3.2 salary band blank                             2729 ->  2365
  3.2 Student / Unemployed band                     2365 ->  2150

decision log:
  1.1 encoding | file read with encoding=cp1252 (1 non-ASCII byte(s) in the file)
  1.2 load | loaded 2814 rows x 65 columns, all as text, default NA parsing off
  1.4 strip | 1 column name(s) had surrounding whitespace and were stripped
  1.5 blanks | 117089 empty/whitespace cells converted to NaN; 35 literal 'None'-style cells kept as text
  1.6 option row | row 0 moved to option_labels (rows 2814 -> 2813)
  1.7 empty rows | 0 fully empty rows found; 0 dropped; 73 

## Written decision log

**Output:** `canonical_2011.csv`, 2,150 rows x 23 columns, nothing one-hot encoded, imputed, merged or scaled. Rows: 2,814 in file -> 2,813 (option-label row) -> 2,729 (84 junk rows) -> 2,365 (364 no salary band) -> 2,150 (215 Student / Unemployed).

**Kept exactly as the file has them:** Country, USState, Age, Industry, OrgSize, DevType, RecommendationActedUpon, ProjectType, OpSys, JobSat, CompTotal (salary band text), PersonalTechSpend12m, SitesFrequentedMost. Blanks stay blank.

**Changed, and why:**
1. `YearsCode`: Excel date damage repaired (`41310 -> 2-5`, `41435 -> 6-10`, `11 -> 11+`).
2. `PurchaseInvolvement`, `PurchaseTypes`, `TechProductsOwned`: one column per option -> one `;` list, the format 2016+ uses.
3. `BudgetOutsideExpenditure`: 26 conflicting rows resolved (Don't know wins; then `>$150,000`; then the highest bracket), stored as one text column.
4. `LanguageHaveWorkedWith`: tick-boxes + 533 write-ins parsed into canonical names (80 distinct), as a `;` list; `LanguageHaveWorkedWith_count` added. `CSS` -> `HTML/CSS`; tools, frameworks and markup are recognised but not counted; the Visual Basic variants are kept apart; two commentary rows decided by hand (listed in Cell 5.3).
5. `ConvertedCompYearly`: band midpoint in annual USD (top band $150,000 by `CONFIG`).
6. Headers renamed to shared names; 10 are confirmed from your list and 13 are proposed (see `header_map_2011.csv`).

**For whoever loads this file:** read it with `pd.read_csv(path, keep_default_na=False, na_values=[""])`. The column `SitesFrequentedMost` holds the real answer `None` (9 rows), which default parsing would turn into missing.

**Left for the pooled pipeline (fit on training years only):** one-hot / multi-hot encoding, rare-level merging, imputation, scaling, feature selection.